# Watching the Moon's shadow cross the North Atlantic — the 12 Aug 2026 total solar eclipse

On **12 August 2026** the Moon's shadow swept across the far north — the Arctic, eastern
**Greenland**, western **Iceland** and the North Atlantic — with greatest eclipse at
**17:47 UTC**. A solar eclipse is one of the few things you can watch *from space*: the Moon
casts a real shadow on Earth and a geostationary satellite sees it directly.

We use **GOES-19 (GOES-East)** full-disk imagery, reprojected to a **map centred on the eclipse
track** (the North Atlantic) so the shadow sweeps through the **middle** of the frame. GOES-East
sits at 75°W, so this reach is toward its eastern limb — a regional map (rather than the full
globe) keeps the eclipse itself front-and-centre.

In [ ]:
import os
import shutil
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from cleopatra.glyphs.gridded.array_glyph import FrameLabel
from IPython.display import Image, display
from loguru import logger
from pyramids.dataset import Dataset
from pyramids.dataset.collection import DatasetCollection
from pyramids.netcdf import NetCDF
from rasterio.crs import CRS
from rasterio.transform import Affine, from_bounds
from rasterio.warp import Resampling, reproject

from earthlens.core import EarthLens

warnings.filterwarnings("ignore")
logger.remove()
plt.rcParams["figure.dpi"] = 80

## How the frames are built

`earthlens`' **`goes`** backend streams raw full-disk ABI granules from the anonymous
`noaa-goes19` S3 bucket (no credentials), one per 10-minute slot from **15:00 to 19:00 UTC**. For
each granule we read the visible bands in their native geostationary grid, synthesise a green
channel (ABI has none: CIMSS `0.45·R + 0.10·veg + 0.45·B`), gamma-stretch to true colour with a
**fixed** stretch (so the shadow reads as real darkening), then **reproject to a lat/lon map** of
the North Atlantic with `rasterio`. Frames are cached, so a re-run resumes.

In [ ]:
SLOTS = pd.date_range("2026-08-12 15:00", "2026-08-12 19:00", freq="10min")
STEP = 4  # decimate the native 5424 px full disk before reprojecting
W, S, E, N = (
    -56.0,
    50.0,
    -8.0,
    73.0,
)  # map window over the eclipse track (Greenland -> Iceland)
OW = 1300
OH = int(round(OW * (N - S) / (E - W)))
DST_CRS = CRS.from_epsg(4326)
DST_T = from_bounds(W, S, E, N, OW, OH)

OUT = Path("out") / "eclipse_goes"
REGION_DIR = OUT / "frames_region"
REGION_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def _geos_georef(nc_path, step):
    """The GOES geostationary CRS + (decimated) transform for a full-disk granule."""
    tmp = os.path.join(tempfile.mkdtemp(), "g.tif")
    NetCDF.read_file(nc_path).get_variable("CMI_C02").to_file(tmp)
    with rasterio.open(tmp) as s:
        t = s.transform
    return s.crs, Affine(t.a * step, t.b, t.c, t.d, t.e * step, t.f)


def render_region(nc_path, dst, step=STEP):
    """True-colour the native full disk, reproject to the North-Atlantic map, save RGB."""
    nc = NetCDF.read_file(nc_path)

    def cmi(name):
        arr = np.squeeze(
            np.asarray(nc.get_variable(name).read_array(unpack=True), dtype="float32")
        )
        return arr[::step, ::step]

    red, veg, blue = (
        cmi("CMI_C02"),
        cmi("CMI_C03"),
        cmi("CMI_C01"),
    )  # 0.64 / 0.86 / 0.47 um
    green = 0.45 * red + 0.10 * veg + 0.45 * blue  # CIMSS synthetic green
    rgb = np.power(np.clip([red, green, blue], 0.0, 1.0), 1 / 2.2)  # (3, H, W), gamma
    src_crs, src_t = _geos_georef(nc_path, step)

    bands = []
    for b in range(3):
        out = np.zeros((OH, OW), dtype="float32")
        reproject(
            source=rgb[b],
            destination=out,
            src_crs=src_crs,
            src_transform=src_t,
            dst_crs=DST_CRS,
            dst_transform=DST_T,
            resampling=Resampling.bilinear,
            src_nodata=0.0,
            dst_nodata=0.0,
        )
        bands.append(out)
    arr = (np.clip(np.stack(bands), 0.0, 1.0) * 255.0).astype("uint8")
    Dataset.create_from_array(
        arr=arr,
        geo=(W, (E - W) / OW, 0.0, N, 0.0, -(N - S) / OH),
        epsg=4326,
        no_data_value=0,
    ).to_file(str(dst))

### Fetch each 10-minute full-disk granule and render the map frame

One granule per slot (~370 MB) is downloaded, the map frame is written, then the granule is
deleted. Cache-aware: a slot whose frame already exists is skipped.

In [ ]:
frames = []  # (timestamp, region_path)
for slot in SLOTS:
    tag = slot.strftime("%Y%m%d%H%M")
    region_path = REGION_DIR / f"{tag}.tif"
    if not region_path.exists():
        granule_dir = Path(tempfile.mkdtemp())
        try:
            job = EarthLens(
                data_source="goes",
                dataset="abi-l2-mcmip",
                satellite="east",  # GOES-19 is GOES-East in 2026
                domain="F",  # full disk, native 10-min scan
                start=slot.strftime("%Y-%m-%d %H:%M"),
                end=(slot + pd.Timedelta(minutes=8)).strftime("%Y-%m-%d %H:%M"),
                fmt="%Y-%m-%d %H:%M",
                lat_lim=[S, N],
                lon_lim=[W, E],
                path=granule_dir,
            )
            got = job.download(progress_bar=False)
            if not got:
                continue  # occasional scan gap at this exact slot -> skip the frame
            render_region(got[0], region_path)
        finally:
            shutil.rmtree(granule_dir, ignore_errors=True)
    frames.append((slot, region_path))

frames = [f for f in frames if f[1].exists()]
print(f"{len(frames)} map frames: {frames[0][0]:%H:%M} -> {frames[-1][0]:%H:%M} UTC")

## The map — the Moon's shadow sweeping the North Atlantic

The eclipse track region, one frame every 10 minutes, played slowly (fps 3). Around greatest
eclipse the penumbra darkens the middle of the frame (Greenland → Iceland → the eastern
Atlantic), then lifts.

In [ ]:
dc = DatasetCollection.from_files(
    [str(f[1]) for f in frames], date_format="%Y%m%d%H%M", date_regex=r"\d{12}"
)
labels = [t.strftime("%H:%M UTC") for t in dc.time]

# Size the figure to the map aspect so it fills the frame edge-to-edge (no letterbox).
glyph = dc.plot(
    rgb_options={"rgb": [0, 1, 2], "surface_reflectance": 255},
    figsize=(12, 12 * OH / OW),
    full_bleed=True,
)
glyph.animate(
    labels,
    interval=350,  # slower playback
    frame_label=FrameLabel(color="white", size=20),
    full_bleed=True,
)
mp4 = OUT / "eclipse_region.mp4"
glyph.save_animation(str(mp4), fps=3, dpi=150, extra_args=["-vf", "scale=1300:-2"])
gif = os.path.join(tempfile.mkdtemp(), "eclipse_region.gif")
glyph.fig.set_dpi(90)
glyph.save_animation(gif, fps=3)
plt.close("all")
display(Image(filename=gif))

The MP4 is written to `out/eclipse_goes/eclipse_region.mp4` for sharing. The dark patch
crossing the middle is the Moon's penumbra — the same shadow that, along the central line from
Greenland through Iceland to Spain, produced totality.